# Explicabilidad (XAI) — Detección de Deepfakes (Fase 3)

**TFM — Máster en Ciencia de Datos e IA (UCM)**

Predecir "real/fake" no basta: el requisito de orientación a Negocio exige
**justificar** la decisión de forma entendible. Lo hacemos en dos planos
complementarios:

1. **Espacial (Grad-CAM)** — mapas de calor sobre el rostro que muestran *qué
   regiones* llevaron al modelo a sospechar manipulación.
2. **Temporal** — la probabilidad de "fake" *fotograma a fotograma* a lo largo
   del clip, que muestra *en qué momentos* aparecen las inconsistencias.

> Requiere haber entrenado el baseline (`models/baseline.pt`) y tener los rostros
> extraídos en `data/interim/` (Fases 1 y 2).


## Preparación del entorno en Colab

Solo actúa en Colab: monta Drive, clona/actualiza el repo e instala dependencias. **Edita `REPO_URL`**. En local, sáltala.

In [ ]:
# --- Bootstrap para Google Colab (en local no hace nada) ---
import os
from pathlib import Path
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["TFM_WORKSPACE"] = "/content/drive/MyDrive/TFM_Deepfake"

    REPO_URL = "https://github.com/TU_USUARIO/TU_REPO.git"   # <-- EDITA ESTO
    PROJECT_ROOT = "/content/TFM_Deepfake_Detection"
    os.environ["TFM_PROJECT_ROOT"] = PROJECT_ROOT
    if not Path(PROJECT_ROOT).exists():
        !git clone {REPO_URL} {PROJECT_ROOT}
    else:
        !cd {PROJECT_ROOT} && git pull -q
    !pip install -q timm grad-cam gradio pyyaml tqdm
    !pip install -q --no-deps facenet-pytorch
    print("Entorno de Colab listo.")

## 0. Configuración

In [ ]:
import os, sys
from pathlib import Path
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
PROJECT_ROOT = Path(os.environ.get("TFM_PROJECT_ROOT", "/content/TFM_Deepfake_Detection")) \
    if IN_COLAB else (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image

from src.utils.seeds import set_seed, load_config, get_device
from src.utils.paths import get_paths

cfg = load_config(PROJECT_ROOT / "config" / "config.yaml")
set_seed(cfg["project"]["seed"])
paths = get_paths(cfg, PROJECT_ROOT)
print("Entorno:", "Colab" if IN_COLAB else "local", "| Device:", get_device())

## 1. Cargar el clasificador por fotograma

Reutilizamos la **cabeza del baseline entrenado** sobre el backbone para formar un
clasificador por frame (imagen de rostro -> probabilidad de fake). Sobre él aplicamos
Grad-CAM. No se entrena nada nuevo.

In [ ]:
from src.explainability.gradcam import load_frame_scorer, GradCAM, overlay_cam_on_image

baseline_ckpt = paths["models"] / "baseline.pt"
assert baseline_ckpt.exists(), (
    f"No encuentro {baseline_ckpt}. Entrena primero el baseline (notebook 02).")

scorer, transform, target_layer, DEVICE = load_frame_scorer(
    cfg["model"]["backbone"], baseline_ckpt)
gradcam = GradCAM(scorer, target_layer)
print("Clasificador por fotograma cargado. Capa objetivo:", type(target_layer).__name__)

## 1b. Localizar un vídeo real y uno manipulado con rostros extraídos

In [ ]:
def list_face_frames(method, video_id, n=None):
    files = sorted((paths["interim"] / method).glob(f"{video_id}_frame*.jpg"))
    return files[:n] if n else files

def first_video_with_faces(method):
    d = paths["interim"] / method
    if not d.exists():
        return None
    files = sorted(d.glob("*_frame*.jpg"))
    return files[0].name.split("_frame")[0] if files else None

real_id = first_video_with_faces("original")
fake_method = cfg["dataset"]["manipulation_methods"][0]
fake_id = first_video_with_faces(fake_method)
print(f"Vídeo REAL: {real_id}")
print(f"Vídeo FAKE: {fake_id}  ({fake_method})")

## 2. Grad-CAM espacial

Para cada fotograma: arriba la imagen original del rostro, abajo el mapa de calor
superpuesto. Las zonas **rojas** son las que más empujaron la decisión hacia "fake".

In [ ]:
def gradcam_grid(method, video_id, n_frames=4, title=""):
    files = list_face_frames(method, video_id, n_frames)
    if not files:
        print("Sin frames para", method, video_id); return
    fig, axes = plt.subplots(2, len(files), figsize=(3 * len(files), 6))
    for i, f in enumerate(files):
        img = Image.open(f).convert("RGB")
        rgb = np.array(img)
        x = transform(img).unsqueeze(0).to(DEVICE)
        cam = gradcam(x)[0]
        with torch.no_grad():
            prob = torch.sigmoid(scorer(x))[0, 0].item()
        over = overlay_cam_on_image(rgb, cam)
        axes[0, i].imshow(rgb); axes[0, i].axis("off"); axes[0, i].set_title(f"frame {i}")
        axes[1, i].imshow(over); axes[1, i].axis("off"); axes[1, i].set_title(f"fake p={prob:.2f}")
    fig.suptitle(title + "  (arriba: original · abajo: Grad-CAM)")
    plt.tight_layout()
    plt.savefig(paths["figures"] / f"gradcam_{method}_{video_id}.png", dpi=120, bbox_inches="tight")
    plt.show()

if fake_id:
    gradcam_grid(fake_method, fake_id, title=f"FAKE ({fake_method})")
if real_id:
    gradcam_grid("original", real_id, title="REAL")

## 3. Explicabilidad temporal

La probabilidad de "fake" frame a frame a lo largo del clip. Picos por encima de
0.5 señalan los instantes donde el modelo detecta inconsistencias; un vídeo real
debería mantenerse bajo de forma estable.

In [ ]:
def temporal_curve(method, video_id, title=""):
    files = list_face_frames(method, video_id)
    if not files:
        print("Sin frames para", method, video_id); return
    probs = []
    for f in files:
        x = transform(Image.open(f).convert("RGB")).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs.append(torch.sigmoid(scorer(x))[0, 0].item())
    plt.figure(figsize=(9, 3))
    plt.plot(range(len(probs)), probs, marker="o")
    plt.axhline(0.5, ls="--", color="gray", label="umbral 0.5")
    plt.ylim(0, 1); plt.xlabel("fotograma"); plt.ylabel("prob. fake")
    plt.title(title); plt.legend(); plt.tight_layout()
    plt.savefig(paths["figures"] / f"temporal_{method}_{video_id}.png", dpi=120, bbox_inches="tight")
    plt.show()

if fake_id:
    temporal_curve(fake_method, fake_id, f"Evolución temporal — FAKE ({fake_method})")
if real_id:
    temporal_curve("original", real_id, "Evolución temporal — REAL")

## 4. Lectura de negocio

Cómo se traduce esto para un equipo no técnico (caso KYC / *onboarding*):

- **El mapa de calor responde al "¿por qué?"**. En lugar de un veredicto opaco, el
  analista ve sobre el rostro las regiones que dispararon la alerta (típicamente
  bordes de la cara, ojos o boca, donde los deepfakes dejan más rastro). Esto da
  *trazabilidad* a la decisión, algo clave en un entorno regulado.
- **La curva temporal aporta confianza**. Un fake suele mostrar inestabilidad
  (picos intermitentes); un vídeo real se mantiene plano y bajo. Es un argumento
  visual e intuitivo para justificar por qué se bloquea o se aprueba una alta.
- **Conexión con el umbral de coste (Fase 2)**. Recuerda que el punto de decisión
  no es 0.5 fijo, sino el umbral que minimiza el coste esperado priorizando no
  dejar pasar deepfakes. La explicabilidad complementa esa política: el umbral
  decide, y el mapa de calor *justifica* ante el cliente o el regulador.

> En la memoria, estas figuras van en la sección de explicabilidad con un par de
> ejemplos (un acierto claro y, si lo hay, un caso dudoso), redactadas para Negocio.


## 5. Conclusiones y advertencia metodológica

- Grad-CAM y la curva temporal hacen el modelo **interpretable**, cumpliendo el
  requisito de que un perfil de Negocio entienda y confíe en la decisión.
- **Advertencia importante:** la calidad de los mapas depende de la calidad del
  modelo. Con un dataset pequeño y un baseline poco entrenado, los mapas ilustran
  el *mecanismo* pero no resaltan artefactos reales. Con el modelo entrenado sobre
  suficientes datos (AUC alto), los mapas se concentran de forma coherente en las
  zonas manipuladas. Repite este notebook tras entrenar con más vídeos.

**Siguiente paso (Fase 4):** productivización — una app en Gradio que reciba un
vídeo y devuelva la predicción junto con el mapa de calor, simulando el entorno real.
